# 安装依赖库

In [ ]:
!pip install mindnlp==0.4.0
!pip uninstall mindformers -y
!pip install transformers==4.40.0
!pip install mindspore==2.4.1

# 导入依赖库

In [ ]:
from mindnlp.transformers import AutoModelForCausalLM, AutoTokenizer
import mindspore as ms
import pandas as pd
import re
import gc

ms.set_context(device_target="Ascend")

# 导入模型与分词器

In [ ]:
model_id = "Qwen/Qwen2.5-3B"
tokenizer = AutoTokenizer.from_pretrained(model_id, ms_dtype=ms.float16, mirror='modelscope')
model = AutoModelForCausalLM.from_pretrained(model_id, ms_dtype=ms.float16, mirror='modelscope')
tokenizer.pad_token_id = tokenizer.eos_token_id

# 辅助函数

### extract_last_number：提取最终结果（数字）

In [ ]:
# 选择最后一个数字作为结果
def extract_last_number(text):
    numbers = re.findall(r'-?\d+(?:\.\d+)?', text)
    return numbers[-1] if numbers else None

### extract_last_assistant_reply：提取第一次大模型生成的回答

In [ ]:
def extract_last_assistant_reply(text):
    if 'assistant\n' in text:
        return text.split('assistant\n')[-1].strip()
    if 'assistant' in text:
        return text.split('assistant')[-1].strip()
    return text.strip()

# 推理函数：predict_gsm8k_batch

In [ ]:
def predict_gsm8k_batch(questions, gold_labels, batch_size=16,output_file="gsm8k_training_results.txt"):
    # 如果文件已存在，先清空内容
    with open(output_file, "w", encoding="utf-8") as f:
        f.write("")

    # questions: 训练集问题
    # gold_labels: 训练集答案
    tokenizer.padding_side = 'left'
    all_predictions = []
    all_outputs = []
    num_questions = len(questions)

    for start_idx in range(0, num_questions, batch_size):
        gc.collect()
        batch_q = questions[start_idx: start_idx + batch_size]
        batch_gold = gold_labels[start_idx: start_idx + batch_size]

        # Step 1: CoT推理
        # messages_step有两部分组成：
        # 1. system类型的消息：确定模型的角色：helpful assistant，与训练数据无关。
        # 2. user类型的消息：用户提问，也就是具体的训练数据。
        # CoT的第一步关键在于：在训练数据（q）的基础上，添加一个提示：Let's think step by step，这样大模型就会开始第一次推理
        messages_step1 = [
            [
                {'role': 'system', 'content': "You are a helpful assistant."},  
                {'role': 'user', 'content': f"{q} Let's think step by step."}   
            ]
            for q in batch_q
        ]

        input_ids_1 = tokenizer.apply_chat_template(
            messages_step1, add_generation_prompt=True, return_tensors="ms", tokenize=True,
            padding=True, truncation=True, max_length=1024
        ) #把消息转化为模型输入
        outputs_1 = model.generate(input_ids_1, max_new_tokens=512, temperature=0.7, top_p=0.95) #输出结果
        step1_texts = tokenizer.batch_decode(outputs_1, skip_special_tokens=True) #解码输出为文本
        step1_replies = [extract_last_assistant_reply(txt) for txt in step1_texts] #提取第一个大模型生成的回复

        # Step 2: 最终答案
        # messages_step2的与messages_step1的区别在于：
        # 训练数据在messages_step1的基础上添加了大模型第一次生成的回答，并在最后添加提示： Therefore, the answer (arabic numerals) is，这样大模型就会开始第二次推理
        messages_step2 = [
            [
                {'role': 'system', 'content': "You are a helpful assistant."},
                {'role': 'user', 'content': f"{q} Let's think step by step. {reply} Therefore, the answer (arabic numerals) is"}
            ]
            for q, reply in zip(batch_q, step1_replies)
        ]
        input_ids_2 = tokenizer.apply_chat_template(
            messages_step2, add_generation_prompt=True, return_tensors="ms", tokenize=True,
            padding=True, truncation=True, max_length=1024
        )#把消息转化为模型输入
        outputs_2 = model.generate(input_ids_2, max_new_tokens=256, temperature=0.7, top_p=0.95)#输出结果
        answer_texts = tokenizer.batch_decode(outputs_2, skip_special_tokens=True)#解码输出为文本
        predictions = [extract_last_number(ans) for ans in answer_texts]#提取最后一个数字作为答案

        # 立即打印当前 batch 的结果
        for q, gold, output, pred in zip(batch_q, batch_gold, answer_texts, predictions):
            print(f"\nQ: {q}\nGold: {gold}\nPred: {pred}\nOutput: {output}\n")
        
        # 将当前 batch 的结果追加写入文件
        with open(output_file, "a", encoding="utf-8") as f:
            for q, gold, output, pred in zip(batch_q, batch_gold, answer_texts, predictions):
                f.write(f"Q: {q}\nGold: {gold}\nPred: {pred}\nOutput: {output}\n\n")
        
        all_outputs.extend(answer_texts)
        all_predictions.extend(predictions)

    return all_outputs, all_predictions

# 评估函数：evaluate_gsm8k

In [ ]:
def evaluate_gsm8k(max_samples=50, batch_size=16, data_path='data/gsm8k.parquet'):
    df = pd.read_parquet(data_path)[:max_samples] # 读入前50条数据
    questions = df['question'].tolist() # 仅用列question作为模型输入
    gold_labels = [row['answer'].split("####")[-1].strip() for _, row in df.iterrows()] # 从 answer 字段中提取“####”之后的内容作为参考答案。
    outputs, predictions = predict_gsm8k_batch(questions, gold_labels, batch_size=batch_size ,output_file="gsm8k_training_results.txt")
    accuracy = sum(pred == gold for pred, gold in zip(predictions, gold_labels)) / len(gold_labels)
    print("GSM8K准确率:", accuracy)#计算推理准确率
    return outputs, gold_labels, accuracy

# 运行函数

In [ ]:
gsm8k_outputs, gsm8k_labels, gsm8k_acc = evaluate_gsm8k(max_samples=50, batch_size=10, data_path='data/gsm8k.parquet')